In [ ]:
import sys
sys.path.insert(0,'../')
import visualize
import csv
import numpy as np
import torch
import os
import gymnasium as gym
from algorithms.ppo import Agent
from algorithms.llm_moral import call_llm_with_state_action,create_llm_env,few_shot_prompt_training

config = visualize.argparser().parse_args(args=[])

In [2]:
def env_runner(env, agent, model_path=None):
    env = gym.wrappers.FlattenObservation(env)
    # Mimic SyncVectorEnv for cleanrl's PPO
    env.single_action_space = env.action_space
    env.single_observation_space = env.observation_space
   

    device = torch.device("cuda" if torch.cuda.is_available() and config.cuda else "cpu")
    agent = agent(env).to(device)

    model_path = model_path or config.model_path
    agent.load_state_dict(torch.load(model_path))

    next_obs, _ = env.reset(seed=config.seed)
    next_obs = torch.Tensor(next_obs).to(device)
    done = False
    steps = 0
    frames = []
    itr = 0

    while not done:
        # action = env.action_space.sample()
        action, logprob, _, value = agent.get_action_and_value(next_obs)
        if config.debug_llm:
            print(env.render())
            state_text, action_text = env.state_as_text()
            actionsets = [frozenset([str(k)]) for k in env.action_mapper.keys()] #TODO: review str casting 
            scenario_prompt = env.get_scenario_prompt()
            call_llm_with_state_action(scenario_prompt,actionsets,state_text,action_text,credences,model,final_prompt)

        state, reward, terminated, truncated, info = env.step(action.cpu().numpy())
        done = np.logical_or(terminated, truncated)
        itr=itr+1

        metrics = env.log()
        # Put each rendered frame into dict for animation
        steps += 1

        frames.append({
            'timestep': steps,
            'frame': env.render(),
            'state': state,
            'action': action,
            'reward': reward,
            'metric_1_name' : metrics['metric1'][0],
            'metric_2_name' : metrics['metric2'][0],
            'metric_1' : metrics['metric1'][1],
            'metric_2' : metrics['metric2'][1]
            }
        )
        next_obs, next_done = torch.Tensor(state).to(device), torch.Tensor(done).to(device)
        
    env.close()
    return frames


## Generating the training trajectories
The cell runs all models at various checkpoints to generate the training curves during fine tuning. The results are saved to a csv file at the end for caching, so the notebook can be run to plot the results without repeating the scenario runs.

In [9]:
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() and config.cuda else "cpu")


stats = []

environments = {"FindMilk": "environments.milk:FindMilk-v4",
                "Driving": "environments.drive:Driving",}

model_folders = {
                #  "GPT-4o-mini": "moral_llm_gpt-4o-mini",
                #  "mistral-nemo": "moral_llm_mistral-nemo",
                #  "synthetic human actions": "RLHF",
                 "base": None}

# for env_name, env_id in environments.items():
#     env = gym.make(env_id, render_mode='ansi', validate=True)
#     env_id = env_id.split(':')[-1] if ':' in env_id else env_id
#     for model, model_folder in model_folders.items():
#         break_flag = False
#         for n in range(5, 1005, 5):
#             # model_path = f"models/Driving_42/moral_llm_gpt-4o-mini/ppo_{n:d}.cleanrl_model"
#             if model=="base":
#                 model_path = f"../models/{env_id}_42/base.cleanrl_model"
#                 break_flag = True
#             else:
#                 model_path = f"../models/{env_id}_42/{model_folder}/ppo_{n:d}.cleanrl_model"
            
#             if not os.path.exists(model_path):
#                 break
#             for i in range(50):
#                 frames = env_runner(env, Agent, model_path=model_path)
#                 data = {"timesteps": frames[-1]['timestep'],
#                         "metric_1_name": frames[-1]["metric_1_name"],
#                         "metric_2_name": frames[-1]["metric_2_name"],
#                         "metric_1": frames[-1]['metric_1'],
#                         "metric_2": frames[-1]['metric_2'],
#                         frames[-1]["metric_1_name"]: frames[-1]['metric_1'],
#                         frames[-1]["metric_2_name"]: frames[-1]['metric_2'],
#                         "seed": config.seed,
#                         "episode": n,
#                         "model": model,
#                         "env_namme": env_name}
#                 stats.append(data)
#                 config.seed += 1
#             if break_flag: break

# SAVE the results
# df = pd.DataFrame(stats)
# df.to_csv("data_learning_curves.csv", index=False)


## Plotting

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("data_learning_curves.csv")
fig, axes = plt.subplots(1,2, layout="constrained")

sns.lineplot(data=df, ax=axes[0], x="episode", y="sleeping babies passed", estimator="mean", hue="model", errorbar=("ci", 95))
sns.lineplot(data=df, ax=axes[1], x="episode", y="crying babies passed", estimator="mean", hue="model", errorbar=("ci", 95))

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1,2, layout="constrained")

sns.lineplot(data=df, ax=axes[0], x="episode", y="Number of car collisions", estimator="mean", hue="model", errorbar=("ci", 95))
sns.lineplot(data=df, ax=axes[1], x="episode", y="Number of grandmas rescued", estimator="mean", hue="model", errorbar=("ci", 95))

# Driving
## Creating a csv from the data

In [ ]:
# stats = []    
# # Specify the CSV file name
# csv_file = 'runs/Driving__ppo__1__moral/kl_div/results.csv'
# # Open the CSV file for writing
# with open(csv_file, mode='w', newline='') as file:
#     writer = csv.DictWriter(file, fieldnames=["modelNumber", "NumberOfCollision", "NumberOfRescuedGrandma"])
    
#     # Write the header
#     writer.writeheader()

#     for j in range(5, 40, 5):
#         modelName = f"ppo_{j}.cleanrl_model"
#         print(f"-------------------Results for model number {j}--------------------")
        
#         ## FOR ROHIT
#         config.model_path = f"runs/Driving__ppo__1__moral/kl_div/{modelName}"

#         for i in range(5):
#             frames = visualize.run(config)
#             stats.append(frames[-1])
#             config.seed += 1
    
#         visualize.print_frames(config.env_id, frames, dt=0.01, indices=[0, len(frames)-1])
#         # metric_1 = 0
#         # metric_2 = 0
#         timesteps = [i['timestep'] for i in stats]
#         metric_1 = [i['metric_1'] for i in stats]
#         metric_2 = [i['metric_2'] for i in stats]
#         print(f'Timesteps: {np.mean(timesteps)} +- {np.std(timesteps)}')
#         print(f'Number of collision: {np.mean(metric_1)} +- {np.std(metric_1)}')
#         print(f'Number of rescued grandma: {np.mean(metric_2)} +- {np.std(metric_2)}')
#         print(f'Number of collision: {metric_1}')
#         print(f'Number of rescued grandma: {metric_2}')
#         row = {
#             "modelNumber": str(j),
#             "NumberOfCollision": f"{np.mean(metric_1):.2f} ± {np.std(metric_1):.2f}",
#             "NumberOfRescuedGrandma": f"{np.mean(metric_2):.2f} ± {np.std(metric_2):.2f}"
#         }
#         # row = f"{j}, +- {np.std(metric_1)},{np.mean(metric_2)} +- {np.std(metric_2)}" 
#         writer.writerow(row)
#         stats.clear()
    

# longest_idx = np.argmax(timesteps)
# print(stats[longest_idx]['frame'])